# ECON 525: Data Management & Visualization in Economics
# Python — Web scraping

Suppose a website lists a team's wins for each year. How could we bring those numbers into Python without copying every row by hand?

**Web scraping** means using code to collect information from web pages. We will begin with a small page we can see in full, then use the same ideas on Wikipedia and a website with hockey statistics.


## Learning objectives

By the end, you should be able to:

- Connect part of a visible web page to the HTML that describes it.
- Find a heading, a link, or a table cell and collect the information inside it.
- Search a website and collect rows from a second results page.
- Check the resulting table and save it for analysis.


## 1. What is HTML?

**HTML** stands for *HyperText Markup Language*. It is text with labels that tell a browser which parts are headings, paragraphs, links, and so on.

For example, this HTML describes one paragraph:

```html
<p>Hockey teams</p>
```

`<p>` starts the paragraph; `</p>` ends it. These labels are called **tags**. The opening tag, the words, and the closing tag together make one **element**.

The browser displays the words **Hockey teams**, without showing the tags. Let's see both versions in the notebook.


### Tools for this lesson

We will use **Beautiful Soup** to read and search HTML, and **MechanicalSoup** to request pages and fill search forms. We also import three small helpers:

- `Path` from `pathlib` reads the practice HTML files saved beside this notebook.
- `HTML` and `display` from `IPython.display` let us show a small HTML page as it would appear to a reader. Later, `display` also shows our tables.
- `urljoin` and `urlencode` from Python's `urllib.parse` help us build web addresses. We will see an example of each when we need it.

If Beautiful Soup or MechanicalSoup is missing, run the following in a separate cell, then restart the kernel and run from the top:

```python
%pip install beautifulsoup4 mechanicalsoup
```

Keep the supplied `scraping-*.html` files with this notebook. We begin with the three small practice pages; the other four are saved copies of real pages for use if the websites are unavailable.


In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlencode
from IPython.display import HTML, display

import time
import pandas as pd
import mechanicalsoup
from bs4 import BeautifulSoup


In [ ]:
paragraph_html = '<p>Hockey teams</p>'
print(paragraph_html)
display(HTML(paragraph_html))


The first output is the HTML text. The second is the displayed paragraph. **The content is the same; we are looking at two representations of it.**

### A small page: code and appearance

Our practice page has a heading, a paragraph, and a numbered list of references. `Path(...).read_text(encoding='utf-8')` reads its file into a Python string; it does not contact a website. UTF-8 tells Python how to read the file's characters.

First, look at the HTML. The `<head>` contains information about the page, such as its browser-tab title. The `<body>` contains the content shown to the reader.


In [ ]:
wiki_html = Path('scraping-wiki-sample.html').read_text(encoding='utf-8')
print(wiki_html)


Now display **that same string** as a page. Find the heading, paragraph, and two numbered references. Then look back at the HTML and find the tags around each one.


In [ ]:
display(HTML(wiki_html))


### Read the labels on the page

`h1` marks the main heading. `p` marks a paragraph. `ol` is an ordered (numbered) list, and each `li` is one list item. An `a` element is a link.

For example:

```html
<a href="https://mechanicalsoup.readthedocs.io/en/stable/">MechanicalSoup documentation</a>
```

The reader sees **MechanicalSoup documentation**. The `href` contains the destination address. An extra piece of information inside an opening tag, such as `href="..."`, is an **attribute**.

Notice that the label and destination are different pieces of information. To create a list of web addresses, we will need the `href`, not just the words a reader sees.


### IDs and classes: labels we can search for

An **ID** identifies one particular element on a page. Our paragraph has `id="summary"`, so we can ask for that paragraph directly.

A **class** is a label that several elements can share. In our page, both reference spans have `class="reference-text"`: they play the same role, even though they contain different references. This is an HTML label, separate from a Python class.

```html
<span class="reference-text">First reference here</span>
<span class="reference-text">Second reference here</span>
```

Think of an ID as one person's name tag and a class as a group label. Here, both elements belong to the group `reference-text`. These labels help us find them; a class name alone does not automatically change their appearance.


### Why are there spans in two different places?

A **`span`** groups a piece of content without making it a new paragraph. For example:

```html
<p>Wins: <span class="wins">31</span></p>
```

The reader sees **Wins: 31** on one line. The `p` marks the whole paragraph; the `span` labels just the number, so code can find it separately.


In [ ]:
display(HTML('<p>Wins: <span class="wins">31</span></p>'))


In our practice page:

- `<h1><span class="mw-page-title-main">Web scraping</span></h1>` is a heading containing a labeled piece of text. The `h1` makes it a heading; the span lets us identify the title text. The long class name follows Wikipedia's title label; it has no special Python meaning.
- `<span class="reference-text">...</span>` groups the source information within a list item. It separates the source link from the nearby **Back to text** link.

Both are spans because both group content. Their **classes and positions** tell us their different jobs. A `p` would mark a paragraph; we do not need a new paragraph just to label part of a heading or list item. Without a class or styling, a span often makes no visible difference.

**Try saying it aloud:** Which tag makes the title a heading? Which attribute helps us locate its text?


## 2. Let Python read the HTML

Right now, `wiki_html` is one long string. To ask questions such as “Which links are inside the references?”, Python needs to recognize the tags and what is inside each one.

**Parsing HTML** means reading those tags and organizing the elements by how they fit inside one another. For example, the source link is inside a span, which is inside a list item, which is inside the references list.

This arrangement is often called a **tree**: the page branches into smaller parts. “Searchable tree” simply means that we can ask for a particular element or search within one branch. This simplified picture follows the heading and **one** reference; our full page also contains the summary and another reference.

![A simplified map of the nested heading and first reference.](scraping-html-tree.png)

`BeautifulSoup(wiki_html, 'html.parser')` creates that organized representation. `html.parser` chooses the HTML reader included with Python. We name the result `soup`; that name is just a variable.


In [ ]:
soup = BeautifulSoup(wiki_html, 'html.parser')


## 3. Find one part, then read what is inside

### Start with a tag: one paragraph

A **selector** is a short description of the element we want. `'p'` means “a paragraph element.” `select_one(...)` returns the **first matching element**, including its tags.

Before running: which paragraph should this find in our page?


In [ ]:
paragraph_tag = soup.select_one('p')
print(paragraph_tag)


The result still has `<p>` and `</p>`. To get just the words, use `.get_text(...)`. The option `strip=True` removes spaces and line breaks from the edges of the text pieces it collects. It does not rewrite the sentence.


In [ ]:
print(paragraph_tag.get_text(strip=True))


### A tag selector: all links

`select(...)` returns a collection of **all** matching elements. In contrast, `select_one(...)` returns just the first match.

`'a'` selects link elements. Predict the number of matches: our page has two **Back to text** links and two documentation links.


In [ ]:
all_links = soup.select('a')
print('Number of links:', len(all_links))
for link in all_links:
    print(link.get_text(strip=True))


### Read a link's destination

The first link says **Back to text**. `.get('href')` reads its `href` attribute instead of its text. The value `#cite-1` points to a place within the same page; it is not an outside source address.


In [ ]:
first_link = all_links[0]
print('Words:', first_link.get_text(strip=True))
print('Destination:', first_link.get('href'))


### A class selector: `.reference-text`

The dot means **has this class**. `'.reference-text'` finds both spans labeled `class="reference-text"`. The dot is part of the selector string; here it does not call a Python method.

Look at the output: do the **Back to text** labels remain?


In [ ]:
reference_spans = soup.select('.reference-text')
for span in reference_spans:
    print(span.get_text(strip=True))


### Combine a tag and a class: `span.mw-page-title-main`

This means “a `span` with class `mw-page-title-main`.” There is no space between the tag and the dot: both conditions describe **the same element**.


In [ ]:
title_tag = soup.select_one('span.mw-page-title-main')
print('Element:', title_tag)
print('Title:', title_tag.get_text(strip=True))


### An ID selector: `#summary`

The `#` means **has this ID**. `'#summary'` finds the element with `id="summary"`. The attribute uses `id="summary"` in HTML; the selector uses `#summary` in Python's quoted string.


In [ ]:
summary_tag = soup.select_one('#summary')
print(summary_tag.get_text(strip=True))


### Search inside something: `ol.references a`

Read the space as **inside**: find `a` elements inside an `ol` with class `references`. They can be several levels down. In our page, a source link is inside `span`, then `li`, then `ol`; it still matches.

Predict: does “inside the references list” alone exclude **Back to text**?


In [ ]:
list_links = soup.select('ol.references a')
for link in list_links:
    print(link.get_text(strip=True))


It does not: both kinds of links are in the list. We need to describe the part of the list that answers our question.

### What if nothing matches?

`select_one(...)` returns `None` when it finds nothing. `select(...)` returns an empty collection. Neither result is a piece of HTML, so check the selector before trying to read text from it.


In [ ]:
missing_tag = soup.select_one('#does-not-exist')
print(missing_tag)
print(soup.select('#does-not-exist'))


### WS1 — Choose what you want to collect · Individual · 5 minutes

Select the main heading using its **tag** (`h1`), then print its text. Next, try selecting `mw-page-title-main` without a dot or tag name. What does Python return, and why?

Use the displayed practice page and its HTML to explain your answer in one sentence.

My explanation: __________________________________________________


In [ ]:
# WS1: select the h1 element and print its text.
# Then try 'mw-page-title-main' as a selector and explain the result.


## 4. Collect the references we actually want

Our question is “Which outside sources does the page cite?” A **Back to text** link takes a reader back to the article, so it should not enter our source list.

Add `span.reference-text` to stay inside the source information. Read `'ol.references span.reference-text a'` as “links inside source spans inside the references list.”

Compare the two counts, then inspect one source's words and address.


In [ ]:
# Broad selector: includes links back to the article as well as source links.
broad_links = soup.select('ol.references a')

# More specific selector: stay inside the source-text spans.
source_tags = soup.select('ol.references span.reference-text a')
print('Broad matches:', len(broad_links))
print('Source matches:', len(source_tags))

# A single worked example: text and URL are different pieces of one tag.
one_source = source_tags[0]
print(one_source.get_text(strip=True))
print(one_source.get('href'))


For each source, `.get('href')` gives an address string. If the attribute is missing, it gives `None`. An `if address:` check lets us keep only nonempty values before adding them to a Python list.


### WS2 — Keep the sources, skip the backlinks · Pairs · 5 minutes

One partner is the **navigator** (explains which tags count), and one is the **driver** (types). Switch roles halfway.

Build a list named `my_reference_urls` containing the `href` values from `source_tags`.
Use an empty list, a `for` loop, and `.append(...)`. Keep only nonempty addresses.
Print the list and its length. Do **not** open the links.

**Checkpoint.** Explain to your partner why a same-page address such as `#cite-1` should not be in a list of outside sources.


In [ ]:
my_reference_urls = []
# TODO (WS2): loop over source_tags.
# TODO: get each href, check it is not empty, and append it to your list.
# TODO: print the list and its length.


## 5. Try the same idea on Wikipedia

Open the [Web scraping article](https://en.wikipedia.org/wiki/Web_scraping) in your browser. Right-click its title and choose **Inspect**. The browser highlights the HTML for the part you clicked. You can also press **F12** and choose **Elements**.

Compare the title's label with our practice page. We used the same class name so that the selector has a familiar target. The real page is much larger, but the question is still small: what is the title, and which source links can we find?

### Getting a page is a separate step

A **server** is the computer providing a website. When we open a web address, our browser sends it a **request**: “Please send this page.” The server sends a **response**, usually containing HTML and a result code.

Beautiful Soup reads HTML we already have. MechanicalSoup can request HTML for us and then use Beautiful Soup to organize it. Its `StatefulBrowser` remembers the current page and address, so we do not have to pass them into every action. Calling the library's class creates an object, which we name `real_browser`; we do not write a new class ourselves.


### Before requesting a website: where is `robots.txt`?

A website owner can place a public text file named **`robots.txt` on the website's server**, at the top level of its address. For example, the file for `https://www.example.org/reports/` would be at:

```text
https://www.example.org/robots.txt
```

This is an example address, not a file on your computer. You can type the address into a browser to read it. A real example is [Wikipedia's robots.txt](https://en.wikipedia.org/robots.txt). Not every website provides one.

A **bot** is a program that visits websites automatically. A **crawler** is a bot that follows links to visit pages. “Crawler rules” just means instructions about which addresses those programs should avoid. For example:

```text
User-agent: *
Disallow: /private/
```

`User-agent: *` applies the rule to all bots. `Disallow: /private/` asks them not to request addresses starting with `/private/` on that website. It does not mean a folder named private on our computer.

These are instructions, not a password lock; the file does not block a program by itself. Our code does not check it automatically. For a project, read it alongside the site's access and reuse rules, collect only what you need, and stop if access is denied. [More about robots.txt](https://developers.google.com/search/docs/crawling-indexing/robots/intro)


### Choose current pages or saved copies

We will normally use `LIVE = True`: Python contacts Wikipedia now, and later the hockey website. If a request fails, read the error, change this to `LIVE = False`, and run from the top. That choice uses copies of the real pages saved on **October 26, 2026**.

The request cells print **LIVE** when they contact a website and **SAVED** when they read one of those files. Saved pages let us practice the same extraction, but they do not tell us what is on the website today.

In the next cell, `soup_config` chooses the same HTML reader we used earlier. `user_agent` identifies our program as a classroom demonstration and gives the course repository as its contact page; [Wikipedia asks automated programs to identify themselves](https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy).


In [ ]:
LIVE = True
real_browser = mechanicalsoup.StatefulBrowser(
    soup_config={'features': 'html.parser'},
    user_agent='ECON525-Classroom-Demo/1.0 (https://github.com/DataManagementVisual/Econ525-Fall2026)'
)

if not LIVE:
    print('Using real website pages saved on October 26, 2026.')


### Request, check, then select

`real_browser.open(wiki_url, timeout=10)` requests the page. The time limit avoids waiting indefinitely for a connection or more response data. The returned `response` contains request results; `response.raise_for_status()` stops with an error for an unsuccessful HTTP result such as 403 (access refused) or 404 (page not found). A `status_code` of 200 means the request succeeded.

`real_browser.page` contains the HTML already organized by Beautiful Soup, so it supports the same `select` and `select_one` methods as `soup`.

In the saved-page branch, `open_fake_page(saved_html, url=wiki_url)` gives MechanicalSoup the saved HTML and its address, without contacting the server. Despite the method's name, these particular saved pages are copies of real website responses.

### Make a complete link address

Some `href` values contain only part of an address, such as `/wiki/Python`. `urljoin` combines that part with the current page address. It only builds a string; it does not visit the link.


In [ ]:
print(urljoin('https://en.wikipedia.org/wiki/Web_scraping', '/wiki/Python'))


Now collect the title and references. Wikipedia labels external source links with class `external`, so we add `.external` to the last `a`. Here it means an `a` with class `external`, just as `.reference-text` found our spans with that class.

**Predict:** does reading an `href` also visit that destination? Watch which line actually makes the request.


In [ ]:
wiki_url = 'https://en.wikipedia.org/wiki/Web_scraping'
if LIVE:
    response = real_browser.open(wiki_url, timeout=10)
    response.raise_for_status()
    print('LIVE:', response.status_code, real_browser.url)
else:
    saved_html = Path('scraping-live-wikipedia.html').read_text(encoding='utf-8')
    real_browser.open_fake_page(saved_html, url=wiki_url)
    print('SAVED:', real_browser.url)

real_wiki_page = real_browser.page
real_title = real_wiki_page.select_one('span.mw-page-title-main')
print('Article title:', real_title.get_text(strip=True))

real_reference_urls = []
for tag in real_wiki_page.select('ol.references span.reference-text a.external'):
    address = tag.get('href')
    if address:
        real_reference_urls.append(urljoin(real_browser.url, address))
print('Reference links:', len(real_reference_urls))
print('First three:', real_reference_urls[:3])


### Why might Python miss something I can see in Chrome?

**JavaScript** is another programming language. A browser such as Chrome can run JavaScript instructions after the initial HTML arrives—for example, to fetch and display a price.

Imagine the server first sends `<p id="price">Loading...</p>`. JavaScript then obtains a price and changes the displayed paragraph to **$25**. Reading only the initial HTML would give us **Loading...**, not the price.

**Neither Beautiful Soup nor MechanicalSoup runs JavaScript.** Beautiful Soup reads the HTML it is given; MechanicalSoup requests pages and uses Beautiful Soup to read them. Neither performs the later JavaScript steps that Chrome can perform. We do not need to learn JavaScript for the pages in this lesson. ([MechanicalSoup FAQ](https://mechanicalsoup.readthedocs.io/en/stable/faq.html))

In Chrome or Edge, **Ctrl + U** shows the HTML originally sent by the server. **Inspect / Elements** shows the page's current structure, including changes made by JavaScript. If visible data is absent from the downloaded HTML, check whether the site offers a data download or an **API**—a documented way for programs to request data directly.


### Check your understanding

Explain these as separate actions: getting a page, finding an element, and reading its text or address. Which of those actions needs a website request?


## 6. A small hockey page: search first, then collect rows

Now imagine we want team names, years, and wins. Start with two small pages containing **fictional teams**, so we can see every row before using the real hockey website.

We will display the first page and inspect the same HTML that Python reads.


In [ ]:
hockey_html_1 = Path('scraping-hockey-page1.html').read_text(encoding='utf-8')
hockey_html_2 = Path('scraping-hockey-page2.html').read_text(encoding='utf-8')
display(HTML(hockey_html_1))


The page contains a search box, a table with two rows, and a **Next page** link. In this local preview, use the form as something to inspect; we will send a real search on the hockey website later.

We create a second MechanicalSoup object, `browser`, for this practice page. `browser.page` holds its organized HTML; `browser.url` holds its address. `open_fake_page` loads our local string. The `.example` address is a classroom label: we will not request that website.


In [ ]:
# StatefulBrowser is the library's class; browser is our particular object.
browser = mechanicalsoup.StatefulBrowser(soup_config={'features': 'html.parser'})

# Load our local HTML into the browser's current-page state. No HTTP request is made.
browser.open_fake_page(hockey_html_1, url='https://classroom.example/hockey/')
print('Current URL label:', browser.url)
print('Page heading:', browser.page.select_one('h1').get_text(strip=True))


### How does a search box become a request?

The displayed search box comes from this HTML:

```html
<form class="form-inline" method="get" action="/hockey/">
  <input name="q" value="">
  <button type="submit">Search</button>
</form>
```

`form` groups the search controls. `input` is the field we type into. Its `name="q"` tells the server which field our answer belongs to; `value` holds that answer. `action="/hockey/"` gives the destination.

Here, `method="get"` means the search values go into the address after a `?`. Searching for `new` gives `?q=new`: “the value of q is new.”

`select_form('form.form-inline')` chooses that form; `browser['q'] = 'new'` fills its `q` field. These steps do not send a request. The selector `'input[name="q"]'` means an input whose `name` attribute equals `q`; square brackets match an attribute's value.

Before filling it, let's see the address helper on its own. `urlencode` turns a dictionary of search values into the text used in an address. It handles spaces and other characters for us.


In [ ]:
print(urlencode({'q': 'new'}))
print(urlencode({'q': 'new york'}))


The `+` in `new+york` represents the space in this search value. Now fill the form and build the complete address. We then load the supplied first page as a practice response. On the real website, `submit_selected(...)` will send the search for us.


In [ ]:
# Select the form by its tag and class; then fill the input whose name is q.
browser.select_form('form.form-inline')
browser['q'] = 'new'
print('Local field value:', browser.page.select_one('input[name="q"]').get('value'))

# Build the URL a GET search would request. urlencode handles spaces and other characters.
form_tag = browser.page.select_one('form.form-inline')
search_path = form_tag.get('action')
search_url = urljoin(browser.url, search_path) + '?' + urlencode({'q': 'new'})
print('A server would receive:', search_url)

# Offline simulation: load the supplied response; this is NOT a real submission.
# On a permitted live site, browser.submit_selected() would send the form.
browser.open_fake_page(hockey_html_1, url=search_url + '&page=1')
print('Saved response rows:', len(browser.page.select('tr.team')))


## 7. From a table row to an observation

Look again at the displayed hockey table. **One team in one year** is one observation. A team in two years should produce two records.

Here is the HTML for the first row:

```html
<tr class="team">
  <td class="name">New York Bears</td>
  <td class="year">1990</td>
  <td class="wins">31</td>
</tr>
```

`tr` is a table row; `td` is a data cell (`th` is a heading cell). `td.name` selects the cell labeled `name`. We search **inside one row** so the name, year, and wins belong to the same observation.

In `'table#teams tr.team'`, `table#teams` finds our table by its ID, and the space asks for team rows inside it. The `#` and dot have the same meanings as before.

First collect the name and wins from just the first row. Text inside HTML is still text in Python: `int('31')` makes it a number we can calculate with.


In [ ]:
team_rows = browser.page.select('table#teams tr.team')
worked_row = team_rows[0]
print(worked_row)
print('Name:', worked_row.select_one('td.name').get_text(strip=True))
wins_text = worked_row.select_one('td.wins').get_text(strip=True)
print('Wins before conversion:', repr(wins_text), type(wins_text))
print('Wins after conversion:', int(wins_text), type(int(wins_text)))


Now put those cells into one dictionary. We use the same conversion for the year. **Predict:** what would happen if we tried `int('Wins')` after selecting a heading cell instead of a data cell?


In [ ]:
worked_record = {
    'team': worked_row.select_one('td.name').get_text(strip=True),
    'year': int(worked_row.select_one('td.year').get_text(strip=True)),
    'wins': int(worked_row.select_one('td.wins').get_text(strip=True)),
}
print(worked_record)


### WS3 — Build the next observation · Groups of 3 · 5 minutes

Roles: **selector detective**, **coder**, and **checker**. Each person explains one part.

Extract the **second** hockey row into `my_record`, with keys `team`, `year`, and `wins`.
Store the name as text and the two numeric values as integers. Print the dictionary.
Python indexing begins at zero: which index selects the second row?

**Checkpoint.** The checker should verify both the team name and the *type* of each numeric value, then explain why these three cells belong together.


In [ ]:
# TODO (WS3): choose the second element of team_rows.
# TODO: build my_record with team (str), year (int), wins (int).
# TODO: print your dictionary and check the numeric types.


### Repeat the same steps for each row

We already know how to make one record. A function lets us reuse those steps on each page: it receives a page, loops over the team rows, and returns a list of dictionaries.

Follow the indentation: `records.append(record)` is inside the loop, so it runs for each row. `return records` is after the loop, so we return all the rows together.

Finally, `pd.DataFrame(records)` uses each dictionary as a row and its keys as column names—the same kind of table we used after SQL.


In [ ]:
def extract_table_data(page):
    """Convert one hockey page into a list of typed team-year records."""
    records = []  # Start with no observations.
    for row in page.select('table#teams tr.team'):
        record = {
            'team': row.select_one('td.name').get_text(strip=True),
            'year': int(row.select_one('td.year').get_text(strip=True)),
            'wins': int(row.select_one('td.wins').get_text(strip=True)),
        }
        records.append(record)  # Add one observation, then continue to the next row.
    return records

first_page_records = extract_table_data(browser.page)
first_page_df = pd.DataFrame(first_page_records)
display(first_page_df)


## 8. What about the next page?

The first page only shows part of the results. **Pagination** means dividing results across pages. Our **Next page** link looks like this:

```html
<a rel="next" href="?q=new&amp;page=2">Next page</a>
```

`rel="next"` labels the link's relationship to the current page. The selector `'a[rel="next"]'` finds an `a` with that attribute value, just as we found the input using its `name`.

The `href` keeps `q=new` and asks for `page=2`. HTML writes `&amp;` for `&`; `.get('href')` gives us the ordinary `&`. This address only contains the part after `?`, so we use `urljoin` again to combine it with the current address.

**Predict:** would page 1 alone give us every team-year in these two practice pages?


### WS4 — Find the next page, then justify when to stop · Solo → pairs · 5 minutes

For the first two minutes, work alone. Select the `a` tag with `rel="next"` from `browser.page`.
Read its `href` and use `urljoin(...)` to store an absolute address in `my_next_url`. Print it.
Then compare with a partner and explain which part keeps the search term `new`.

**Checkpoint.** Your code should construct an address, without fetching it. What should a scraper do when there is no Next link?


In [ ]:
# TODO (WS4): select 'a[rel="next"]' from browser.page.
# TODO: read its href, combine it with browser.url, and print my_next_url.
# Stop rule in your own words: ______________________________


### Combine the two pages and check

We load the supplied second page, extract its rows with the same function, and join the two lists with Python's `+`.

`duplicated(['team', 'year'])` marks a repeated team-year with `True`; `.sum()` counts those repeats. New York Bears in 1990 and in 1991 are different observations. Two copies of New York Bears in **1990** would be a repeated observation.

The second page has no Next link. Predict what `select_one('a[rel="next"]')` will return there.


In [ ]:
# Find the next address from the current page, independently of your activity answer.
next_tag = browser.page.select_one('a[rel="next"]')
if next_tag is None:
    raise ValueError('Expected a Next link on the first practice page.')
next_url = urljoin(browser.url, next_tag.get('href'))

# Offline navigation: substitute the supplied page-2 response for a network call.
browser.open_fake_page(hockey_html_2, url=next_url)
second_page_records = extract_table_data(browser.page)
all_records = first_page_records + second_page_records  # + joins two Python lists.
teams_df = pd.DataFrame(all_records)

print('Page 1 rows:', len(first_page_records), '| Page 2 rows:', len(second_page_records))
print('Duplicate team-year observations:', teams_df.duplicated(['team', 'year']).sum())
print('Next link after page 2:', browser.page.select_one('a[rel="next"]'))
display(teams_df)


### Keep the table

Check the four rows against the two practice pages. Are the names correct? Are year and wins numeric? Is each team-year unique?

Save with `to_csv(..., index=False)` to leave out pandas' row-number column, then read the file back. Rerunning replaces this practice CSV. `browser.close()` finishes the browser session, much as we close a SQL connection; our table and CSV remain available.


In [ ]:
teams_df.to_csv('hockey-practice.csv', index=False)
display(pd.read_csv('hockey-practice.csv'))

browser.close()


## 9. Search the real hockey website

Open [Hockey Teams on Scrape This Site](https://www.scrapethissite.com/pages/forms/). This is a website created for practicing web scraping, with historical hockey statistics. It is larger than our fictional example, but it also has a search form, table rows, and numbered pages.

Search for **new** in your browser. Inspect a team-name cell and the search field. Then click page **2** and compare the addresses. We will repeat those actions in Python, using our earlier `LIVE` choice.

First, open the page and print the form's HTML. We use `real_browser` again, so its current page changes from Wikipedia to hockey.


In [ ]:
hockey_url = 'https://www.scrapethissite.com/pages/forms/'
if LIVE:
    time.sleep(1)
    response = real_browser.open(hockey_url, timeout=10)
    response.raise_for_status()
    print('LIVE:', response.status_code, real_browser.url)
else:
    saved_html = Path('scraping-live-hockey-initial.html').read_text(encoding='utf-8')
    real_browser.open_fake_page(saved_html, url=hockey_url)
    print('SAVED:', real_browser.url)

print(real_browser.page.select_one('form.form-inline'))


### Fill the search box, then send it

As in the practice page, `select_form` chooses the form and `['q']` fills its named field. `submit_selected(timeout=10)` then sends the search and returns a response. We check that response before using its page.

Look for `q=new` in the printed address. The saved-page option loads the response for **this same search**; changing the search word would require a new live request, rather than reusing that saved file.


In [ ]:
real_browser.select_form('form.form-inline')
real_browser['q'] = 'new'

if LIVE:
    time.sleep(1)
    response = real_browser.submit_selected(timeout=10)
    response.raise_for_status()
    print('LIVE:', response.status_code, real_browser.url)
else:
    saved_html = Path('scraping-live-hockey-search.html').read_text(encoding='utf-8')
    real_browser.open_fake_page(saved_html, url=hockey_url + '?q=new')
    print('SAVED:', real_browser.url)

real_hockey_search = real_browser.page


### Collect the table rows

Inspect one real row before assuming its labels match our practice page. The site uses `tr.team`, with cells `td.name`, `td.year`, and `td.wins`. It does **not** use our table ID `teams`, so we remove `table#teams` from the row selector.

The remaining steps stay the same. This function returns records from one page; we will reuse it for page 2. Compare the first displayed record with the website.


In [ ]:
def extract_real_hockey(page):
    records = []
    for row in page.select('tr.team'):
        records.append({
            'team': row.select_one('td.name').get_text(strip=True),
            'year': int(row.select_one('td.year').get_text(strip=True)),
            'wins': int(row.select_one('td.wins').get_text(strip=True)),
        })
    return records

real_first_records = extract_real_hockey(real_hockey_search)
display(pd.DataFrame(real_first_records).head())


### Follow page 2

Find the link whose displayed text is **2** among the links inside `.pagination`. The loop below stops at that link. If none is found, it raises a readable error rather than trying to follow a missing element.

Why use the number? When preparing this example, the site's Next arrow pointed back to page 1 after a search. Reading the actual address is safer than trusting the arrow alone. Our target should keep `q=new` and contain `page_num=2`.

`follow_link(...)` requests the selected link and updates the current page. `requests_kwargs={'timeout': 10}` passes our time limit to that request. After extracting page 2, we check repeated team-year pairs: a successful request can still bring back a page we already collected.


In [ ]:
real_page2_link = None
for link in real_browser.page.select('.pagination a'):
    if link.get_text(strip=True) == '2':
        real_page2_link = link
        break

if real_page2_link is None:
    raise ValueError('Page 2 link missing: inspect the pagination HTML.')
real_page2_url = urljoin(real_browser.url, real_page2_link.get('href'))
print('Page 2 URL:', real_page2_url)

if LIVE:
    time.sleep(1)
    response = real_browser.follow_link(real_page2_link, requests_kwargs={'timeout': 10})
    response.raise_for_status()
    print('LIVE:', response.status_code, real_browser.url)
else:
    saved_html = Path('scraping-live-hockey-page2.html').read_text(encoding='utf-8')
    real_browser.open_fake_page(saved_html, url=real_page2_url)
    print('SAVED:', real_browser.url)

real_second_records = extract_real_hockey(real_browser.page)
real_teams_df = pd.DataFrame(real_first_records + real_second_records)
print('Rows:', len(real_teams_df))
print('Repeated team-year keys:', real_teams_df.duplicated(['team', 'year']).sum())
display(real_teams_df.head())


### Save and check what we collected

We requested only two results pages, so this is a sample of the website's data. Check the row count and repeated team-year pairs above, then save the table as `hockey-two-pages.csv`. This replaces that demonstration CSV if you run it again.

For another website, begin the same way: look at its displayed page, inspect the relevant HTML, and test one row before collecting more. The website's labels may differ, even when the displayed table looks similar.


In [ ]:
real_csv_path = 'hockey-two-pages.csv'
real_teams_df.to_csv(real_csv_path, index=False)
print('Saved:', real_csv_path)
real_browser.close()


### WS5 — Explain it to a partner · Optional · 3 minutes

Point to one line that requests a website page and one line that reads a value from HTML. Explain what each line gives us.

Why do we check repeated team-year pairs after following page 2? If the output says **SAVED**, what do we know about when the page was collected?


In [ ]:
# Optional WS5: write your explanation here. No new scraping requests are needed.
# Real request line: ______________________________________________
# Extraction line: ________________________________________________
# Repeated rows and saved-page date: ____________________________


## What should you remember now?

**Get the HTML → parse it → select the parts you want → extract their values → check and save.**

Parsing organizes the tags. Selecting finds particular elements. Extracting reads their text or attributes. Getting a live page is the step that contacts a website; the other steps work on HTML we already have.

![From a displayed page through HTML and selected values to a table.](scraping-pipeline.png)

**Exit question:** Why could extracting every link, or collecting only the first page, give a table that does not answer your research question?

My explanation: __________________________________________________


### References

The Wikipedia and hockey examples draw on [ScrapingBee's MechanicalSoup tutorial](https://www.scrapingbee.com/blog/getting-started-with-mechanicalsoup/).

- [MDN: basic HTML syntax](https://developer.mozilla.org/en-US/docs/Learn_web_development/Core/Structuring_content/Basic_HTML_syntax) and [the span element](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/span).
- [Beautiful Soup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/): finding elements and reading text and attributes.
- [MechanicalSoup tutorial](https://mechanicalsoup.readthedocs.io/en/stable/tutorial.html): requesting pages, forms, and navigation.
- [Python's URL helpers](https://docs.python.org/3/library/urllib.parse.html) and [IPython display tools](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html).


### Beyond HTML — optional references

We focus on **HTML** in this lesson. Websites can also provide data as **JSON** (named fields and lists, often from an API), **XML** (tagged data, for example a data feed), or **PDF** documents. We will not cover extracting these formats here.

For later: [MDN's introduction to JSON](https://developer.mozilla.org/en-US/docs/Learn_web_development/Core/Scripting/JSON), [Python's XML tools](https://docs.python.org/3/library/xml.etree.elementtree.html), and [pypdf's text-extraction guide](https://pypdf.readthedocs.io/en/stable/user/extract-text.html). Each format needs an appropriate reading method; our HTML selectors are for HTML elements.
